In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

In [ ]:
def load_data(file):
    
    hdul = fits.open(file)
    
    primary_hdu = hdul[0]
    data_hdu = hdul[1]

    with fits.open(file) as hdul:
        pheader = hdul[0].header
        if str(file).endswith("raw.fits"):
            data = hdul[1].data
        else:
            data = hdul[0].data
        dheader = hdul[1].header
    return data, pheader, dheader    

In [ ]:
def load_file(filelist):
    for file in filelist:
        data, pheader, dheader = load_data(file)
    return data, pheader, dheader

In [ ]:
here = pathlib.Path('.')

raw_files = list(here.rglob('**/*raw.fits*'))
mask_files = list(here.rglob('**/*maskfile.fits*'))
blev_files = list(here.rglob('**/*blevfile.fits*'))
bias_files = list(here.rglob('**/*biasfile.fits*'))
dark_files = list(here.rglob('**/*darkfile.fits*'))
flat_files = list(here.rglob('**/*flatfile.fits*'))
print(raw_files)

In [ ]:
raw_data, raw_pheader, raw_dheader = load_file(raw_files)
mask_data, mask_pheader, mask_dheader = load_file(mask_files)
bias_data, bias_pheader, bias_dheader = load_file(bias_files)
dark_data, dark_pheader, dark_dheader = load_file(dark_files)
flat_data, flat_pheader, flat_dheader = load_file(flat_files)

In [ ]:
pc_data = raw_data[0].copy()
mask_copy = mask_data[0].copy()
# demonstrator's notes; use the PC chip
vmin = pc_data.min()
vmax = pc_data.max()
bad_pixel = np.sum(mask_copy == 256)


print(bad_pixel)

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(9, 9))
ax1.imshow(pc_data, origin='lower', vmin=vmin, vmax=vmax*0.1)
ax2.imshow(mask_copy, origin='lower')

From the mask data, it doesn't seem like the mask file are that influencial, I'll ignore them for now

In [ ]:
exptime = raw_pheader['EXPTIME']
gain = raw_pheader['ATODGAIN']

In [ ]:
dark_scaled = dark_data * exptime
first_cal = raw_data - bias_data - dark_scaled

In [ ]:
unfilter_data = raw_data[0].copy()
filter_data = first_cal[0].copy()
# demonstrator's notes; use the PC chip
vmin_u = unfilter_data.min()
vmax_u = unfilter_data.max()
vmin_f = filter_data.min()
vmax_f = filter_data.max()


fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(9, 9))
ax1.imshow(unfilter_data, origin='lower', vmin=vmin_u, vmax=vmax_u*0.1)
ax2.imshow(filter_data, origin='lower', vmin=vmin_f, vmax=vmax_f*0.1)

In [ ]:
adj_flat = flat_data / 100.0
corrected_flat = np.where(flat_data > 0, first_cal / adj_flat, 0)
final_cal = (corrected_flat * gain) / exptime

In [ ]:
unfilter_data = raw_data[0].copy()
filter_data = final_cal[0].copy()
# demonstrator's notes; use the PC chip
vmin_u = unfilter_data.min()
vmax_u = unfilter_data.max()
vmin_f = filter_data.min()
vmax_f = filter_data.max()


fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(9, 9))
ax1.imshow(unfilter_data, origin='lower', vmin=vmin_u, vmax=vmax_u*0.1)
ax2.imshow(filter_data, origin='lower', vmin=vmin_f, vmax=vmax_f*0.1)